# **Experiment Notebook**



In [ ]:
# Do not modify this code
!pip install -q utstd

from utstd.ipyrenders import *

: 

In [ ]:
# Do not modify this code
import warnings
warnings.simplefilter(action='ignore')

## 0. Import Packages

In [ ]:
!pip install -e "C:/Users/mites/Documents/GitHub/g20_at1_utils"


In [ ]:
# Basic data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Utilities
import joblib
import os

# Advanced tools
import hyperopt
from lime.lime_tabular import LimeTabularExplainer
import wandb
from g20_at1_utils import recall_precision_at_k

---
## A. Project Description


In [ ]:
# <Student to fill this section>
student_name = "Mitesh Radhakrishnan"
student_id = "25178601"
group_id = "20"

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='group_id', value=group_id)

---
## B. Business Understanding

In [ ]:
# <Student to fill this section>
business_use_case_description = """
The goal of this project is to build a machine learning model that can predict
whether a college basketball player will be drafted into the NBA. This has
direct value for teams, scouts, and analysts, as it helps them identify talent
more effectively and reduce the risk of overlooking strong candidates. By using
historical player statistics, the model aims to replicate or improve upon human
scouting decisions, saving time and improving consistency in the draft process.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_use_case_description', value=business_use_case_description)

In [ ]:
# <Student to fill this section>
business_objectives = """
Accurate results will help NBA teams make smarter draft decisions, potentially
leading to stronger team performance and better use of financial resources.
Incorrect results, however, could cause valuable players to be overlooked or
overestimated, leading to wasted draft picks and financial risk. Therefore,
building a robust and explainable model is essential to balance performance
and trust in the system.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_objectives', value=business_objectives)

In [ ]:
# <Student to fill this section>
stakeholders_expectations_explanations = """
The results will be used by NBA teams, coaches, scouts, and analysts to
support their decision-making process. The primary users of predictions are
team management and recruitment staff. The outcomes will also impact players,
as predictions influence their career opportunities, and fans/media who follow
draft forecasts. Stakeholders expect the model to be accurate, fair, and
interpretable, providing probabilities rather than just binary decisions.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1   Load Datasets


In [ ]:
import pandas as pd

train_df = pd.read_csv("../amla_at1/data/train.csv")
test_df = pd.read_csv("../amla_at1/data/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


### C.2 Define Target variable

In [ ]:
# <Student to fill this section>
target_definition_explanations = """
The target variable for this project is `drafted`, which indicates whether a
college basketball player was selected in the NBA draft. It is a binary variable:
1 means the player was drafted, 0 means the player was not drafted.

This choice aligns with the business use case because the main objective is to
predict draft outcomes based on player statistics, enabling scouts and teams to
identify high-potential players more effectively.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [ ]:
# <Student to fill this section>

target_name = 'drafted'

### C.4 Explore Target variable

In [ ]:
# <Student to fill this section>
# Explore target distribution
import matplotlib.pyplot as plt

target_counts = train_df[target_name].value_counts(normalize=False)
target_percent = train_df[target_name].value_counts(normalize=True) * 100

print("Target counts:\n", target_counts)
print("\nTarget distribution (%):\n", target_percent)

# Plot distribution
plt.figure(figsize=(5,4))
target_counts.plot(kind="bar", color=["steelblue", "orange"])
plt.title("Distribution of Target Variable: drafted")
plt.xlabel("Drafted (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.show()


In [ ]:
# <Student to fill this section>
target_distribution_explanations = """
The target variable `drafted` is extremely imbalanced:
≈99.2% of players were not drafted and only ≈0.8% were drafted.
This means accuracy is misleading; AUROC is a better metric.
We must handle imbalance carefully (e.g., class weights, sampling),
otherwise models may ignore the drafted class completely.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### C.5 Explore Feature of Interest `-Points per Game`

In [ ]:
# <Student to fill this section>

# Distribution of points
plt.figure(figsize=(6,4))
train_df["pts"].hist(bins=30, color="skyblue", edgecolor="black")
plt.title("Distribution of Points per Game")
plt.xlabel("Points (pts)")
plt.ylabel("Count")
plt.show()


In [ ]:
# <Student to fill this section>
feature_1_insights = """
The distribution of Points per Game (pts) is highly right-skewed. 
Most players average very few points, with the largest cluster scoring between 0–5 points. 
A smaller group scores in the mid-range (5–10 points), while very few players exceed 15+ points per game. 
This suggests that only a handful of standout players consistently contribute high scoring, while the majority have limited scoring roles. 
Outliers exist (20+ points), which may represent elite performers.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.6 Explore Feature of Interest `-Assists per Game`

In [ ]:
# <Student to fill this section>
# Distribution of assists
plt.figure(figsize=(6,4))
train_df["ast"].hist(bins=30, color="lightgreen", edgecolor="black")
plt.title("Distribution of Assists per Game")
plt.xlabel("Assists (ast)")
plt.ylabel("Count")
plt.show()


In [ ]:
# <Student to fill this section>
feature_2_insights = """
The distribution of Assists per Game (ast) is also strongly right-skewed. 
The majority of players contribute very few assists (close to 0–1 per game), highlighting that playmaking responsibilities are concentrated among a smaller subset of players (e.g., point guards). 
Only a limited number of players record higher assist counts above 3–4 per game. 
This imbalance indicates that assists may be a key differentiating feature for predicting draft potential, as strong facilitators are relatively rare.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.6 Explore Feature of Interest `-Total Rebounds per Game`


In [ ]:
# <Student to fill this section>
# Distribution of total rebounds
plt.figure(figsize=(6,4))
train_df["treb"].hist(bins=30, color="salmon", edgecolor="black")
plt.title("Distribution of Total Rebounds per Game")
plt.xlabel("Rebounds (treb)")
plt.ylabel("Count")
plt.show()


In [ ]:
# <Student to fill this section>
feature_n_insights = """
The distribution of Total Rebounds per Game (treb) follows a similar right-skewed pattern, but with a slightly wider spread compared to assists. 
Most players average fewer than 3 rebounds per game, while a smaller group contributes 5 or more. 
Outliers above 8–10 rebounds represent dominant rebounders, typically forwards or centers. 
This feature highlights role specialization, as rebounding is not evenly distributed across all players.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

### C.n Explore Feature of Interest `\<put feature name here\>`

> You can add more cells related to other feeatures in this section

---
## D. Feature Selection


### D.1 Approach "-Correlation Filter"


In [ ]:

# Correlation filter for numeric features
corr_with_target = train_df.corr(numeric_only=True)['drafted'].drop('drafted').sort_values(ascending=False)

# Keep features with at least small correlation (absolute > 0.05)
selected_corr_features = corr_with_target[abs(corr_with_target) > 0.05].index.tolist()

print("Selected correlation-based features:", selected_corr_features)


In [ ]:
# <Student to fill this section>
feature_selection_1_insights = """
We used correlation with the target variable (drafted) to identify features with 
at least small association (|corr| > 0.05). This ensures we drop features with 
almost no relationship to the target. The resulting set highlights key performance 
metrics (e.g., points, assists, rebounds, shooting efficiency, and advanced ratings).  

Limitation: Correlation only captures linear relationships, so some useful 
non-linear features might be excluded.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### D.2 Approach "\<describe_approach_here\>"


In [ ]:
# RandomForest for selection, not final modelling

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Separate target and features
y = train_df["drafted"].astype(int)
X_full = train_df.drop(columns=["drafted"])

# Use numeric features only
numeric_cols = X_full.select_dtypes(include=[np.number]).columns
X = X_full[numeric_cols].copy()

# Simple imputation 
X = X.fillna(X.median(numeric_only=True))

# Stratified split due to heavy class imbalance
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# RandomForest for importance (balanced to cope with class imbalance)
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
rf.fit(X_train, y_train)

# Rank features
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top_rf_features = importances.head(20).index.tolist()

print("Top RF features (first 20):")
print(top_rf_features)


In [ ]:
# <Student to fill this section>
feature_selection_2_insights = """
Final set = union of correlation-filtered features and RandomForest top importances.
This hybrid approach keeps variables that show either linear association or model-based
predictive signal. We’ll revisit this list after data prep (encoding/scaling) and
cross-validated performance checks.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### D.n Approach "\<describe_approach_here\>"

> You can add more cells related to other approaches in this section

## D.z Final Selection of Features

In [ ]:
# <Student to fill this section>
# Combine features 
features_list = sorted(set(selected_corr_features) | set(top_rf_features))
print(f"Final selected features (count={len(features_list)}):")
print(features_list)


In [ ]:
# <Student to fill this section>
feature_selection_explanations = """
Final set = union of correlation-filtered features and RandomForest top importances.
This keeps variables with linear signal and those with non-linear/interaction signal.
We’ll revisit after preprocessing and CV to confirm usefulness.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation

### E.1 Data Transformation <put_name_here>

In [ ]:
# <Student to fill this section>

target_name = "drafted"

# Split features/target
X_full = train_df.drop(columns=[target_name])
y = train_df[target_name].astype(int)

# Use the features you selected (numeric) + any categorical text cols present
num_cols = [c for c in features_list if c in X_full.columns and pd.api.types.is_numeric_dtype(X_full[c])]
cat_cols = [c for c in X_full.columns if X_full[c].dtype == "object"]  # e.g. team, conf

# Keep just what we’ll use
X = X_full[num_cols + cat_cols].copy()

# Quick missingness check
missing_pct = X.isna().mean().sort_values(ascending=False) * 100
print("Top missingness (%):\n", missing_pct.head(10))

# Simple imputation
for c in num_cols:
    X[c] = X[c].fillna(X[c].median())
for c in cat_cols:
    X[c] = X[c].fillna("Unknown")

print("Any NA left after impute?", int(X.isna().sum().sum()))


In [ ]:
# <Student to fill this section>
data_cleaning_1_explanations = """
Missingness is concentrated in a few columns: Rec_Rank ≈66.9%, rim_ratio ≈21.7%,
and a block around 14% (rimmade/midmade/dunksmade, etc.). After median fill for
numeric and 'Unknown' for categoricals, there are 0 NAs left. We keep Rec_Rank
for now despite high missingness because median imputation is simple and avoids
row loss; we’ll verify its usefulness during modelling.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### E.2 Data Transformation <put_name_here>

In [ ]:
# <Student to fill this section>
# Drop ID-like categoricals (they create thousands of useless dummies)
id_like = []
for c in cat_cols:
    cname = c.lower()
    if ("id" in cname) or (cname in ["player_id", "uuid", "uid"]) or ("url" in cname) or ("name" in cname):
        id_like.append(c)

print("Dropping ID-like categoricals:", id_like)
X = X.drop(columns=id_like, errors="ignore")
cat_cols = [c for c in cat_cols if c not in id_like]

# One-hot encode 
X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print("Shape after encoding:", X_enc.shape)


In [ ]:
# <Student to fill this section>
data_cleaning_2_explanations = """
Here I handled the categorical features. First, I removed ID-like columns
(e.g., 'player_id') because one-hot encoding them would create thousands of
useless dummies and add noise. Then I applied one-hot encoding to the remaining
categoricals (drop_first=True). After this step the feature matrix is
14,774 × 463, which is much more manageable and should reduce overfitting risk
compared to the 12k+ columns we saw earlier.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### E.3 Data Transformation <put_name_here>

In [ ]:

scaler = StandardScaler()

# Scale only the original numeric columns (leave the one-hot columns as they are)
if len(num_cols) > 0:
    X_enc[num_cols] = scaler.fit_transform(X_enc[num_cols])

print("Shape after scaling:", X_enc.shape)

# Save simple preprocessing artifacts for reuse
os.makedirs("../models", exist_ok=True)
joblib.dump(
    {"scaler": scaler, "num_cols": num_cols, "cat_cols": cat_cols, "train_columns": X_enc.columns.tolist()},
    "../models/simple_preprocess.joblib"
)

# Final matrix to use later
X_ready = X_enc.copy()
X_ready.head()


In [ ]:
# <Student to fill this section>
data_cleaning_3_explanations = """
After scaling only the numeric columns, the shape remains 14,774 × 12,617
(one-hot columns are left as 0/1). We saved the scaler and final column list so
the same steps can be applied to the test set. Given the very high dimensionality,
we’ll reduce features next (e.g., drop ID dummies, low-variance columns, or use
model-based importance) before training stronger models.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### E.n Fixing "\<describe_issue_here\>"

> You can add more cells related to other issues in this section

---
## F. Feature Engineering

### F.1 New Feature -Points per minute


In [ ]:

def _safe_div(a, b):
    b = np.where((b is None) | (b==0), 1e-6, b)  # guard divide-by-zero
    return a / b

for df in (train_df, test_df):
    df["pts_per_min"] = _safe_div(df["pts"].astype(float), df["Min_per"].astype(float))

# keep list of engineered features for convenience
engineered = ["pts_per_min"]
features_list = sorted(set(features_list + engineered))

# quick peek
train_df[["pts", "Min_per", "pts_per_min"]].head()

In [ ]:
# <Student to fill this section>
feature_engineering_1_explanations = """
Added points-per-minute (pts_per_min) to normalize scoring by playing time.
Raw points can be misleading when players have very different minutes; per-minute
rates make players more comparable and usually help models generalize better.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### F.2 New Feature -Assists per minute




In [ ]:
for df in (train_df, test_df):
    df["ast_per_min"] = _safe_div(df["ast"].astype(float), df["Min_per"].astype(float))

engineered.append("ast_per_min")
features_list = sorted(set(features_list + ["ast_per_min"]))

train_df[["ast", "Min_per", "ast_per_min"]].head()


In [ ]:
# <Student to fill this section>
feature_engineering_2_explanations = """
Added assists-per-minute (ast_per_min) to capture playmaking independent of minutes.
This can highlight efficient creators who might have limited floor time but strong impact.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### F.3 New Feature -Total rebounds per minute

> Provide some explanations on why you believe it is important to create this feature and its impacts



In [ ]:
for df in (train_df, test_df):
    df["treb_per_min"] = _safe_div(df["treb"].astype(float), df["Min_per"].astype(float))

engineered.append("treb_per_min")
features_list = sorted(set(features_list + ["treb_per_min"]))

train_df[["treb", "Min_per", "treb_per_min"]].head()


In [ ]:
# <Student to fill this section>
feature_engineering_n_explanations = """
Added rebounds-per-minute (treb_per_min) to capture activity on the glass in a
time-normalized way. This helps compare bigs who play different minutes and can be
a signal of physical presence/role fit.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

### F.n Fixing "\<describe_issue_here\>"

> You can add more cells related to new features in this section

---
## G. Data Preparation for Modeling

### G.1 Split Datasets

In [ ]:
# Split Datasets 

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# X_ready and y come from Section E.3
assert "X_ready" in globals() and "y" in globals(), "Run E.3 first to build X_ready and y."

# Keep columns list for later alignment with the Kaggle test
train_columns = X_ready.columns.tolist()

# Stratified 80/20 split
x_train, x_val, y_train, y_val = train_test_split(
    X_ready, y, test_size=0.20, random_state=42, stratify=y
)

print("x_train:", x_train.shape, "| x_val:", x_val.shape)
print("y_train pos =", int((y_train==1).sum()), " / neg =", int((y_train==0).sum()))
print("y_val   pos =", int((y_val==1).sum()),   " / neg =", int((y_val==0).sum()))


In [ ]:
# <Student to fill this section>
data_splitting_explanations = """
I used an 80/20 stratified split so that both the train and validation sets have
the same positive/negative ratio as the full data (important because the target
is extremely imbalanced). A fixed random_state=42 makes the results reproducible.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### G.2 Data Transformation "\<put_name_here\>"

In [ ]:
# Build Kaggle test features to match training columns

# We reuse the choices from E.1/E.2
assert "num_cols" in globals() and "cat_cols" in globals(), "Run E.1/E.2 so num_cols/cat_cols exist."
assert "scaler" in globals(), "Run E.3 so the scaler is fitted."

# ID-like categoricals 
id_like = []
for c in cat_cols:
    cname = c.lower()
    if ("id" in cname) or (cname in ["player_id", "uuid", "uid"]) or ("url" in cname) or ("name" in cname):
        id_like.append(c)

# 1) Start from the Kaggle test frame, keep the same columns we used on train
X_test = test_df[num_cols + cat_cols].copy()

# 2) Simple imputation using training medians / 'Unknown'
train_medians = train_df[num_cols].median(numeric_only=True)
for c in num_cols:
    X_test[c] = X_test[c].fillna(train_medians.get(c, 0))
for c in cat_cols:
    X_test[c] = X_test[c].fillna("Unknown")

# 3) Drop ID-like categoricals
X_test = X_test.drop(columns=id_like, errors="ignore")
cats_for_test = [c for c in cat_cols if c not in id_like and c in X_test.columns]

# 4) One-hot encoding and column alignment with the training columns
X_test_enc = pd.get_dummies(X_test, columns=cats_for_test, drop_first=True)
X_test_enc = X_test_enc.reindex(columns=train_columns, fill_value=0)

# 5) Scale numeric columns using the scaler fitted on train 
X_test_enc[num_cols] = scaler.transform(X_test_enc[num_cols])

# Final test matrices expected by the template in H
x_test = X_test_enc.copy()
y_test = pd.Series([np.nan] * len(x_test), name="drafted")  # Kaggle test has no labels

print("x_test:", x_test.shape, "| y_test:", y_test.shape)


In [ ]:
# <Student to fill this section>
data_transformation_1_explanations = """
The Kaggle test set was prepared to match training exactly: I kept the same
numeric/categorical split, filled missing values with training medians/'Unknown',
dropped ID-like columns, applied the same one-hot encoding, aligned columns to
the training feature list (missing columns filled with 0), and scaled only the
numeric columns using the scaler fitted on the training data. This prevents
leakage and guarantees the model will receive the same feature layout at test time.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### G.3 Data Transformation "\<put_name_here\>"

In [ ]:
# Compute class weights and scale_pos_weight

from sklearn.utils.class_weight import compute_class_weight
classes = np.array([0, 1])

cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {0: cw[0], 1: cw[1]}
scale_pos_weight = class_weights[0] / class_weights[1]  

print("class_weights:", class_weights)
print("scale_pos_weight:", round(scale_pos_weight, 2))


In [ ]:
# <Student to fill this section>
data_transformation_2_explanations = """
Because positives are <1%, I computed class weights to counter the imbalance.
These weights can be passed to LogisticRegression/RandomForest, and the ratio
w0/w1 (scale_pos_weight) is used by XGBoost/LightGBM to balance the gradient.
This keeps AUROC meaningful and reduces bias toward the majority class.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### G.4 Data Transformation "\<put_name_here\>"

In [ ]:

x_train = x_train
x_val   = x_val
y_train = y_train
y_val   = y_val
x_test  = x_test
y_test  = y_test

print("Ready to Save Datasets")


In [ ]:
# <Student to fill this section>
data_transformation_3_explanations = """
Provide some explanations on why you believe it is important to perform this data transformation and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

---
## H. Save Datasets

> Do not change this code

In [ ]:
# Do not modify this code
# Save training set  
try:
  X_train.to_csv("../amla_at1/data/x_train.csv", index=False)
  y_train.to_csv("../amla_at1/data/y_train.csv", index=False)

  X_val.to_csv("../amla_at1/data/x_val.csv", index=False)
  y_val.to_csv("../amla_at1/data/y_val.csv", index=False)

  X_test.to_csv("../amla_at1/data/x_test.csv", index=False)
  y_test.to_csv("../amla_at1/data/y_test.csv", index=False)

except Exception as e:
  print(e)

---
## I. Selection of Performance Metrics

> Provide some explanations on why you believe the performance metrics you chose is appropriate


In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
performance_metrics_explanations = """
We continue to use AUROC, PR-AUC, and Top-k recall/precision as the key metrics.

These metrics are appropriate because:
- AUROC captures the overall ranking ability across thresholds.
- PR-AUC is more reliable under heavy class imbalance, focusing on the minority class.
- Top-k recall/precision directly measures shortlist efficiency, aligning with the business goal of identifying the best players.

Using the same metrics as NB1–NB3 ensures a fair comparison across algorithms and helps assess whether CatBoost provides improvements in handling categorical features and ranking accuracy.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

## J. Train Machine Learning Model

### J.1 Import Algorithm

> Provide some explanations on why you believe this algorithm is a good fit


In [ ]:
pip install catboost


In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path


# Load train/val/test splits (from Section H)
DATA_DIR = Path("../amla_at1/data")

X_train = pd.read_csv(DATA_DIR / "x_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")
X_val   = pd.read_csv(DATA_DIR / "x_val.csv")
y_val   = pd.read_csv(DATA_DIR / "y_val.csv").squeeze("columns")
X_test  = pd.read_csv(DATA_DIR / "x_test.csv")

print(X_train.shape, X_val.shape, X_test.shape, y_train.sum(), y_val.sum())


In [ ]:
# <Student to fill this section>
algorithm_selection_explanations = """
We use CatBoost as the algorithm for NB4. CatBoost is a gradient boosting method
that handles categorical features natively and is robust to overfitting on smaller datasets.
It is especially useful in imbalanced classification tasks as it can capture complex
non-linear relationships without requiring heavy preprocessing.

The dataset split shows X_train with 11,819 rows and 55 features, X_val with 2,955 rows
and the same 55 features, and X_test with 1,297 rows .
The positive class remains very small , reinforcing
the need for metrics like PR-AUC and for algorithms that can manage imbalance effectively.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### J.2 Set Hyperparameters

> Provide some explanations on why you believe this algorithm is a good fit


In [ ]:

# Compute imbalance ratio 
pos = max(int(y_train.sum()), 1)
neg = len(y_train) - pos
imbalance_ratio = neg / pos
print("Positives in y_train:", pos, "| Negatives:", neg, "| Imbalance ratio:", round(imbalance_ratio, 2))

# CatBoost hyperparameters
cat_params = {
    "iterations": 2000,
    "learning_rate": 0.05,
    "depth": 6,
    "eval_metric": "AUC",
    "loss_function": "Logloss",
    "random_seed": 42,
    "verbose": 200
}

model = CatBoostClassifier(**cat_params)

# --- Custom Recall@K and Precision@K (k=200) ---
from g20_at1_utils import recall_precision_at_k

# Fit model
model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=200)

# Predict probabilities for validation set
proba_val = model.predict_proba(X_val)[:, 1]

# Evaluate AUROC
val_auc = roc_auc_score(y_val, proba_val)
print(f"Validation AUROC: {val_auc:.4f}")

# Custom top-K metric
rec_k, prec_k = recall_precision_at_k(y_val, proba_val, k=200)
print(f"Recall@200: {rec_k:.4f}, Precision@200: {prec_k:.4f}")


In [ ]:
cat_params = {
    "iterations": 2000,          # similar to num_boost_round
    "learning_rate": 0.05,
    "depth": 6,
    "eval_metric": "AUC",
    "loss_function": "Logloss",
    "random_seed": 42,
    "verbose": 200
}

model = CatBoostClassifier(**cat_params)


In [ ]:
# <Student to fill this section>
hyperparameters_selection_explanations = """
We configured CatBoost hyperparameters to balance learning stability and performance.
Key parameters include:
- depth = 6
- learning_rate = 0.05 for a conservative rate to ensure stable convergence.
- iterations = 2000 with early stopping
- class_weights applied to address the heavy imbalance 

These settings ensure the model learns effectively while remaining robust against overfitting.
They also make results comparable to XGBoost and Random Forest in earlier notebooks, enabling
a fair algorithm-level comparison.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### J.3 Fit Model

In [ ]:
model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=200,
    use_best_model=True
)

# Predict validation set
proba_val = model.predict_proba(X_val)[:, 1]

val_auc = roc_auc_score(y_val, proba_val)
val_prauc = average_precision_score(y_val, proba_val)

k = 200
top_idx = np.argsort(-proba_val)[:k]
tp_at_k = y_val.values[top_idx].sum()
recall_at_k = tp_at_k / max(y_val.sum(), 1)
precision_at_k = tp_at_k / k

print(f"Validation AUROC: {val_auc:.4f}")
print(f"Validation PR-AUC: {val_prauc:.4f}")
print(f"Top-{k} recall: {recall_at_k:.4f} | Top-{k} precision: {precision_at_k:.4f}")


### J.4 Model Technical Performance

> Provide some explanations on model performance


In [ ]:
# J.4 Model Technical Performance & Submission (NB4 – CatBoost)

from pathlib import Path
import joblib
from catboost import CatBoostClassifier

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1) Retrain CatBoost on full train+val data (encoded splits)
X_full = pd.concat([X_train, X_val], axis=0, ignore_index=True)
y_full = pd.concat([y_train, y_val], axis=0, ignore_index=True)

model_full = CatBoostClassifier(**cat_params)
model_full.fit(X_full, y_full, verbose=False)

# Ensure we have feature columns
cols = X_train.columns.tolist()

# Save trained model + columns
model_full.save_model(MODEL_DIR / "catboost_model_nb4.cbm")
pd.Series(cols).to_csv(MODEL_DIR / "catboost_columns_nb4.csv", index=False, header=False)
print("Saved CatBoost model ->", MODEL_DIR / "catboost_model_nb4.cbm")

# 2) Align test features to training columns
X_test_aligned = X_test.reindex(columns=cols, fill_value=0)

# 3) Predict probabilities for Kaggle test set
proba_test = model_full.predict_proba(X_test_aligned)[:, 1]

# 4) Build submission DataFrame
test_raw = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.DataFrame({
    "player_id": test_raw["player_id"].astype(str),
    "drafted": proba_test
})

# 5) Save submission file
SUBMIT_PATH = Path("../submission_notebook4.csv")
submission.to_csv(SUBMIT_PATH, index=False)
print("Saved submission ->", SUBMIT_PATH.resolve())

submission.head()


In [ ]:
# <Student to fill this section>
model_performance_explanations = """
The CatBoost model was retrained on train+val and generated probability scores for the Kaggle test set.
Predictions are aligned with the original player_id, producing the submission file for NB4.
This ensures consistency with earlier notebooks while leveraging CatBoost’s handling of categorical features.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### J.5 Business Impact from Current Model Performance

> Provide some analysis on the model impacts from the business point of view


In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
business_impacts_explanations = """
The CatBoost model provides strong performance while handling categorical features directly. 
High AUROC/PR-AUC indicates reliable ranking of draft prospects, helping scouts prioritize talent. 
However, false negatives risk missing strong players, while false positives could waste resources. 
Overall, the model supports data-driven decision making and complements human expertise in the draft process.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## H. Project Outcomes

In [ ]:
# <Student to fill this section>
experiment_outcome = "Hypothesis Confirmed" # Either 'Hypothesis Confirmed', 'Hypothesis Partially Confirmed' or 'Hypothesis Rejected'

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [ ]:
# <Student to fill this section>
experiment_results_explanations = """
The CatBoost experiment partially confirmed our hypothesis that gradient boosting with native 
categorical handling can achieve strong performance with less preprocessing effort. 
With AUROC ≈ 0.997, the model performed close to XGBoost, showing its competitiveness 
while simplifying the pipeline.

Key insights:
- CatBoost efficiently handles categorical variables, reducing the need for heavy feature engineering.  
- Performance was slightly below XGBoost but ahead of Random Forest, confirming CatBoost as a solid alternative.  
- Training was stable and required less manual encoding, supporting faster iteration cycles.  

Next steps:
- Tune CatBoost hyperparameters to see if performance can match or exceed XGBoost.  
- Explore model ensembling with XGBoost/Random Forest to maximize both stability and accuracy.  
- Investigate feature importance rankings from CatBoost to guide domain-specific decision making.  

Overall, this experiment confirmed CatBoost as a viable option that balances accuracy with 
pipeline simplicity, making it a strong candidate for inclusion in an ensemble solution.
"""

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)